In [3]:

# EXPERIMENT NO: 9
# TEXT CLASSIFICATION AND TEXT GENERATION USING TRANSFORMERS
# Student Name: Dhivya A
# Roll No: 24BAD020

!pip -q install transformers datasets accelerate sentencepiece scikit-learn
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("EXPERIMENT NO: 9")
print("Student Name: Dhivya A")
print("Roll No: 24BAD020")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

# ==================================================
# PART A: DATASET PREPARATION AND PREPROCESSING
# ==================================================
print("\nPART A: DATASET PREPARATION")
# Load a small IMDb dataset subset
data = load_dataset("stanfordnlp/imdb")
train_data = data["train"].shuffle(seed=42).select(range(200))
test_data = data["test"].shuffle(seed=42).select(range(100))
print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))
print("\nSample review:", train_data[0]["text"][:200])
print("Label:", train_data[0]["label"], "(0=Negative, 1=Positive)")

# Load lightweight DistilBERT tokenizer
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)
train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)
train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")
columns = ["input_ids", "attention_mask", "labels"]
train_data.set_format("torch", columns=columns)
test_data.set_format("torch", columns=columns)

print("Text preprocessing completed!")

# ==================================================
# PART B: TEXT CLASSIFICATION
# ==================================================
print("\nPART B: TEXT CLASSIFICATION")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

def metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0)
    }

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
    compute_metrics=metrics
)

print("Training DistilBERT...")
trainer.train()

results = trainer.evaluate()
print("\nEvaluation Results:")
for key, value in results.items():
    if isinstance(value, (int, float)):
        print(key, ":", round(value, 4))

# Test new reviews
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        output = model(**inputs)
    label = output.logits.argmax(dim=-1).item()
    return "Positive" if label == 1 else "Negative"

print("\nNew Review Predictions:")
for review in [
    "This movie was wonderful and very entertaining.",
    "This movie was boring and terrible."
]:
    print("Review:", review)
    print("Prediction:", predict_sentiment(review))

# Save classification model
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")
print("Classification model saved!")

# ==================================================
# PART C: TEXT GENERATION USING T5-SMALL
# ==================================================
print("\nPART C: TEXT GENERATION")

t5_name = "google-t5/t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_model.to(device)

texts = [
    "Artificial intelligence helps computers perform tasks that normally require human intelligence. It is used in healthcare, education, agriculture and transportation.",
    "Renewable energy comes from sunlight, wind and water. It reduces pollution and helps protect the environment."
]

generated_results = []

for text in texts:
    inputs = t5_tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output = t5_model.generate(
            **inputs, max_new_tokens=35, num_beams=2
        )

    summary = t5_tokenizer.decode(output[0], skip_special_tokens=True)
    print("\nInput:", text)
    print("Generated Summary:", summary)

    generated_results.append({
        "Input Text": text,
        "Generated Summary": summary
    })

pd.DataFrame(generated_results).to_csv(
    "generated_summaries.csv", index=False
)

# ==================================================
# PART D: TESTING AND OUTPUT ANALYSIS
# ==================================================
print("\nPART D: OUTPUT ANALYSIS")

print("Classification model: DistilBERT")
print("Generation model: T5-small")
print("Classification metrics:", {
    k: round(results[k], 4)
    for k in ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]
    if k in results
})

print("\nModel Comparison:")
print("Text Classification: Assigns labels such as Positive or Negative.")
print("Text Generation: Produces text such as a summary.")
print("Classification model: DistilBERT")
print("Text generation model: T5-small")

print("\nRESULT:")
print("Text classification and text generation were implemented.")



EXPERIMENT NO: 9
Student Name: Dhivya A
Roll No: 24BAD020
Device: CPU

PART A: DATASET PREPARATION
Training samples: 200
Testing samples: 100

Sample review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F
Label: 1 (0=Negative, 1=Positive)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Input: Artificial intelligence helps computers perform tasks that normally require human intelligence. It is used in healthcare, education, agriculture and transportation.
Generated Summary: artificial intelligence helps computers perform tasks that normally require human intelligence. it is used in healthcare, education, agriculture and transportation.

Input: Renewable energy comes from sunlight, wind and water. It reduces pollution and helps protect the environment.
Generated Summary: renewable energy comes from sunlight, wind and water.

PART D: OUTPUT ANALYSIS
Classification model: DistilBERT
Generation model: T5-small
Classification metrics: {'eval_accuracy': 0.6, 'eval_precision': 0.6, 'eval_recall': 0.4468, 'eval_f1': 0.5122}

Model Comparison:
Text Classification: Assigns labels such as Positive or Negative.
Text Generation: Produces text such as a summary.
Classification model: DistilBERT
Text generation model: T5-small

RESULT:
Text classification and text generation were i